In [1]:
import os
import gc
import json
import time
import math
import warnings
from contextlib import nullcontext

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy import io
from tqdm import tqdm

warnings.filterwarnings("ignore")

try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    HAS_XLA = True
except Exception:
    torch_xla = None
    xm = None
    HAS_XLA = False

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

IS_TPU = False
if HAS_XLA:
    try:
        device = xm.xla_device()
        IS_TPU = True
    except Exception:
        IS_TPU = False

if not IS_TPU:
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
        torch.backends.cudnn.benchmark = True
        torch.backends.cudnn.deterministic = False
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DEVICE_KIND = "tpu" if IS_TPU else ("cuda" if device.type == "cuda" else "cpu")


def move_to_device(x):
    return x.to(device, non_blocking=(DEVICE_KIND == "cuda"))


def maybe_mark_step():
    if IS_TPU:
        try:
            torch_xla.sync()
        except Exception:
            xm.mark_step()


def save_checkpoint(obj, path):
    if IS_TPU:
        xm.save(obj, path)
    else:
        torch.save(obj, path)


def load_checkpoint(path):
    return torch.load(path, map_location="cpu")


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print(f"PyTorch version : {torch.__version__}")
print(f"Device kind     : {DEVICE_KIND}")
print(f"Device          : {device}")
if DEVICE_KIND == "cuda":
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif DEVICE_KIND == "tpu":
    print("TPU runtime detected via torch_xla")
else:
    print("CPU runtime detected (debug mode)")

PyTorch version : 2.10.0+cu128
Device kind     : cuda
Device          : cuda
GPU             : Tesla T4
VRAM            : 15.6 GB


In [2]:
# -----------------------------
# Paths and competition settings
# -----------------------------
COMP_ROOT    = "/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2"
RAW_PATH     = os.path.join(COMP_ROOT, "raw")
TEST_IN_PATH = os.path.join(COMP_ROOT, "test_in")
STATS_PATH   = os.path.join(COMP_ROOT, "stats", "feat_min_max.mat")

EXP_DIR          = "/kaggle/working/experiments/phase2_competent"
CKPT_DIR         = os.path.join(EXP_DIR, "checkpoints")
LOG_DIR          = os.path.join(EXP_DIR, "logs")
MASK_DIR         = os.path.join(EXP_DIR, "episode_masks")
TOPK_JSON        = os.path.join(LOG_DIR, "topk_checkpoints.json")
TRAIN_LOG_JSON   = os.path.join(LOG_DIR, "train_log.json")
STATS_JSON       = os.path.join(EXP_DIR, "zscore_stats.json")
PRED_SAVE_PATH   = "/kaggle/working/preds.npy"

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(MASK_DIR, exist_ok=True)

MONTHS    = ["APRIL_16", "JULY_16", "OCT_16", "DEC_16"]
TIME_IN   = 10
TIME_OUT  = 16
HORIZON   = TIME_IN + TIME_OUT
S1, S2    = 140, 124
VAL_FRAC  = 0.15

MET_VARS  = ["cpm25", "q2", "t2", "u10", "swdown", "pblh", "v10", "psfc", "rain"]
EMI_VARS  = ["PM25", "NH3", "SO2", "NOx", "NMVOC_e", "NMVOC_finn", "bio"]
ALL_FEATS = MET_VARS + EMI_VARS
V         = len(ALL_FEATS)
U10_IDX   = ALL_FEATS.index("u10")

# -----------------------------
# Training and model hyper-params
# -----------------------------
EPOCHS       = 40
LR           = 7e-4
WEIGHT_DECAY = 3e-3
DROPOUT      = 0.25
N_LEVELS     = 2

if IS_TPU:
    UNET_BASE_CH = 48
    LSTM_HIDDEN  = 48
    TRAIN_BATCH_SIZE = 2
    TARGET_EFFECTIVE_BS = 16
    NUM_WORKERS = 0
    PIN_MEMORY = False
elif DEVICE_KIND == "cuda":
    UNET_BASE_CH = 40
    LSTM_HIDDEN  = 40
    TRAIN_BATCH_SIZE = 2
    TARGET_EFFECTIVE_BS = 8
    NUM_WORKERS = 2
    PIN_MEMORY = True
else:
    UNET_BASE_CH = 24
    LSTM_HIDDEN  = 24
    TRAIN_BATCH_SIZE = 1
    TARGET_EFFECTIVE_BS = 4
    NUM_WORKERS = 0
    PIN_MEMORY = False

GRAD_ACCUM_STEPS = max(1, (TARGET_EFFECTIVE_BS + TRAIN_BATCH_SIZE - 1) // TRAIN_BATCH_SIZE)
VAL_BATCH_SIZE = 1
TEST_BS = 1
PERSISTENT_WORKERS = NUM_WORKERS > 0
AMP_ENABLED = DEVICE_KIND == "cuda"

EARLY_STOP_PATIENCE = 6
TOPK_BEST = 3
RESUME_IF_AVAILABLE = True

# Stage-2 fine-tune (best-checkpoint restart with lower LR)
ENABLE_STAGE2_FINETUNE = True
STAGE2_EPOCHS = 8
STAGE2_LR = 2e-4
STAGE2_PATIENCE = 4

# Geophysics-aware augmentation/TTA controls.
# Horizontal flips are optional and require u10 sign correction when used.
USE_HFLIP_AUG = True
USE_FLIP_TTA = True

# Episode mask construction from helper notebook definition
EPISODE_MASK_MODE = "official"   # "official" or "fast"
EPISODE_PERIOD = 24
EPISODE_STD_MULT = 3.0
EPISODE_PM_MIN = 1.0

# Loss blending (proxy for hidden weighted score)
W_RMSE = 0.30
W_GLOBAL_SMAPE = 0.25
W_EPISODE_SMAPE = 0.30
W_EPISODE_CORR = 0.15

# Penalize over-prediction outside episodic cells.
W_NON_EP_OVERSHOOT = 0.05

print("Configuration loaded")
print(f"  Features           : {V} ({len(MET_VARS)} met + {len(EMI_VARS)} emission)")
print(f"  Time in/out        : {TIME_IN}/{TIME_OUT}")
print(f"  Spatial            : {S1}x{S2}")
print(f"  Device             : {DEVICE_KIND}")
print(f"  Train/Val/Test BS  : {TRAIN_BATCH_SIZE}/{VAL_BATCH_SIZE}/{TEST_BS}")
print(f"  Grad accum steps   : {GRAD_ACCUM_STEPS}")
print(f"  Effective BS       : {TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"  Model channels     : base={UNET_BASE_CH}, lstm={LSTM_HIDDEN}, levels={N_LEVELS}")
print(f"  LR/WD/Dropout      : {LR}/{WEIGHT_DECAY}/{DROPOUT}")
print(f"  Early stop patience: {EARLY_STOP_PATIENCE}")
print(f"  Top-K checkpoints  : {TOPK_BEST}")
print(f"  Episode mode       : {EPISODE_MASK_MODE}")
print(f"  Flip aug / flip TTA: {USE_HFLIP_AUG} / {USE_FLIP_TTA}")
print(f"  Stage2 FT          : {ENABLE_STAGE2_FINETUNE} | epochs={STAGE2_EPOCHS} | lr={STAGE2_LR}")
print(
    f"  Loss weights       : rmse={W_RMSE}, gSMAPE={W_GLOBAL_SMAPE}, "
    f"eSMAPE={W_EPISODE_SMAPE}, eCorr={W_EPISODE_CORR}, nonEpOver={W_NON_EP_OVERSHOOT}"
)

Configuration loaded
  Features           : 16 (9 met + 7 emission)
  Time in/out        : 10/16
  Spatial            : 140x124
  Device             : cuda
  Train/Val/Test BS  : 2/1/1
  Grad accum steps   : 4
  Effective BS       : 8
  Model channels     : base=40, lstm=40, levels=2
  LR/WD/Dropout      : 0.0007/0.003/0.25
  Early stop patience: 6
  Top-K checkpoints  : 3
  Episode mode       : official
  Flip aug / flip TTA: True / True
  Stage2 FT          : True | epochs=8 | lr=0.0002
  Loss weights       : rmse=0.3, gSMAPE=0.25, eSMAPE=0.3, eCorr=0.15, nonEpOver=0.05


In [3]:
# -----------------------------
# Stats + episode masks + datasets
# -----------------------------
def compute_zscore_stats(raw_path, months, features, out_path, sample_stride=5):
    stats = {}
    print("Computing z-score stats (sampled) ...")
    for feat in features:
        chunks = []
        for month in months:
            arr = np.load(os.path.join(raw_path, month, f"{feat}.npy"), mmap_mode="r")
            chunks.append(arr[::sample_stride].ravel())
        flat = np.concatenate(chunks).astype(np.float64)
        stats[feat] = {
            "mean": float(flat.mean()),
            "std": float(flat.std()) + 1e-8,
        }
        print(f"  {feat:<12} mean={stats[feat]['mean']:>10.4f} std={stats[feat]['std']:>10.4f}")

    with open(out_path, "w") as f:
        json.dump(stats, f)
    return stats


if os.path.exists(STATS_JSON):
    with open(STATS_JSON, "r") as f:
        DATA_STATS = json.load(f)
    print(f"Loaded z-score stats from {STATS_JSON}")
else:
    DATA_STATS = compute_zscore_stats(RAW_PATH, MONTHS, ALL_FEATS, STATS_JSON, sample_stride=5)

PM_MEAN = float(DATA_STATS["cpm25"]["mean"])
PM_STD = float(DATA_STATS["cpm25"]["std"])


try:
    from statsmodels.tsa.seasonal import STL
except Exception:
    STL = None

if EPISODE_MASK_MODE == "official" and STL is None:
    raise ImportError("statsmodels is required for official episode mask mode. Install statsmodels or switch EPISODE_MASK_MODE='fast'.")


def compute_episode_mask_for_month(month, force_rebuild=False):
    out_fp = os.path.join(MASK_DIR, f"{month}_episode_mask.npy")
    if os.path.exists(out_fp) and not force_rebuild:
        return np.load(out_fp)

    cpm = np.load(os.path.join(RAW_PATH, month, "cpm25.npy")).astype(np.float32)
    Tm, Hm, Wm = cpm.shape

    if EPISODE_MASK_MODE == "official":
        data_2d = cpm.reshape(Tm, -1)
        mask_2d = np.zeros_like(data_2d, dtype=np.uint8)

        for k in tqdm(range(data_2d.shape[1]), desc=f"Episode mask STL {month}", leave=False):
            ts = data_2d[:, k].astype(np.float64)
            stl = STL(ts, period=EPISODE_PERIOD, robust=True)
            res = stl.fit()
            resid = res.resid
            std_val = float(np.std(resid) + 1e-8)
            mask_2d[:, k] = ((resid > EPISODE_STD_MULT * std_val) & (ts > EPISODE_PM_MIN)).astype(np.uint8)

        mask = mask_2d.reshape(Tm, Hm, Wm)
    else:
        # Fast approximation for iteration speed: per-hour seasonal baseline.
        hod = np.arange(Tm) % EPISODE_PERIOD
        seasonal = np.zeros_like(cpm, dtype=np.float32)
        for h in range(EPISODE_PERIOD):
            idx = np.where(hod == h)[0]
            if len(idx) == 0:
                continue
            seasonal[idx] = cpm[idx].mean(axis=0, keepdims=True)

        resid = cpm - seasonal
        std_val = resid.std(axis=0, keepdims=True) + 1e-8
        mask = ((resid > EPISODE_STD_MULT * std_val) & (cpm > EPISODE_PM_MIN)).astype(np.uint8)

    np.save(out_fp, mask)
    print(f"Saved episode mask for {month} -> {out_fp} | ratio={mask.mean():.5f}")
    return mask


def load_timeline_and_episode_masks():
    feat_arrs = []
    for feat in ALL_FEATS:
        month_arrs = []
        for month in MONTHS:
            arr = np.load(os.path.join(RAW_PATH, month, f"{feat}.npy")).astype(np.float32)
            month_arrs.append(arr)
        feat_arrs.append(np.concatenate(month_arrs, axis=0))

    timeline_np = np.stack(feat_arrs, axis=1).astype(np.float32)  # (T_total, V, H, W)

    month_masks = []
    for month in MONTHS:
        month_masks.append(compute_episode_mask_for_month(month))
    episode_np = np.concatenate(month_masks, axis=0).astype(np.float32)  # (T_total, H, W)

    return timeline_np, episode_np


timeline_np, episode_np = load_timeline_and_episode_masks()

for vi, feat in enumerate(ALL_FEATS):
    timeline_np[:, vi] = (timeline_np[:, vi] - float(DATA_STATS[feat]["mean"])) / float(DATA_STATS[feat]["std"])

timeline = torch.from_numpy(timeline_np)
episode_mask = torch.from_numpy(episode_np)

# Free numpy copies after torch views are created
del timeline_np, episode_np
gc.collect()

total_T = timeline.shape[0]
total_starts = total_T - HORIZON + 1

n_val = max(1, int(total_starts * VAL_FRAC))
val_start_min = total_starts - n_val
train_max_start = max(0, val_start_min - HORIZON)  # gap to avoid train/val overlap

train_starts = np.arange(0, train_max_start + 1, dtype=np.int64)
val_starts = np.arange(val_start_min, total_starts, dtype=np.int64)


class Phase2Dataset(Dataset):
    def __init__(self, starts, split="train"):
        self.starts = starts
        self.split = split

    def __len__(self):
        return len(self.starts)

    def __getitem__(self, idx):
        s = int(self.starts[idx])

        hist = timeline[s:s + TIME_IN].contiguous()                        # (T_in, V, H, W)
        targ = timeline[s + TIME_IN:s + HORIZON, 0].contiguous()           # (T_out, H, W)
        emsk = episode_mask[s + TIME_IN:s + HORIZON].contiguous()          # (T_out, H, W)

        if self.split == "train" and USE_HFLIP_AUG and np.random.rand() > 0.5:
            hist = hist.flip(dims=(-1,))
            # Horizontal mirror changes the sign of zonal wind (u10).
            hist[:, U10_IDX] = -hist[:, U10_IDX]
            targ = targ.flip(dims=(-1,))
            emsk = emsk.flip(dims=(-1,))

        return hist, targ, emsk


train_ds = Phase2Dataset(train_starts, split="train")
val_ds = Phase2Dataset(val_starts, split="val")

loader_kwargs = dict(
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
)
if NUM_WORKERS > 0:
    loader_kwargs["prefetch_factor"] = 2

train_loader = DataLoader(
    train_ds,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    drop_last=(len(train_ds) >= TRAIN_BATCH_SIZE),
    **loader_kwargs,
)
val_loader = DataLoader(
    val_ds,
    batch_size=VAL_BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    **loader_kwargs,
)

ram_gb = timeline.numel() * timeline.element_size() / 1e9
print(f"Timeline tensor shape: {tuple(timeline.shape)} | RAM ~ {ram_gb:.2f} GB")
print(f"Episode mask shape   : {tuple(episode_mask.shape)} | ratio={float(episode_mask.mean()):.5f}")
print(f"Train samples        : {len(train_ds)}")
print(f"Val samples          : {len(val_ds)}")
print(f"Train batches        : {len(train_loader)}")
print(f"Val batches          : {len(val_loader)}")

h_b, y_b, m_b = next(iter(train_loader))
print(f"hist batch   : {tuple(h_b.shape)} (B, T_in, V, H, W)")
print(f"target batch : {tuple(y_b.shape)} (B, T_out, H, W)")
print(f"mask batch   : {tuple(m_b.shape)} (B, T_out, H, W)")

Computing z-score stats (sampled) ...
  cpm25        mean=   33.5755 std=   52.2020
  q2           mean=    0.0115 std=    0.0070
  t2           mean=  291.5968 std=   14.0679
  u10          mean=    1.5474 std=    3.5459
  swdown       mean=  221.6683 std=  309.1720
  pblh         mean=  763.6564 std=  635.6765
  v10          mean=    0.1381 std=    2.9944
  psfc         mean=87986.7203 std=17645.5063
  rain         mean=    0.1369 std=    1.9813
  PM25         mean=    0.0000 std=    0.0000
  NH3          mean=    0.0000 std=    0.0000
  SO2          mean=    0.0000 std=    0.0000
  NOx          mean=    0.0000 std=    0.0000
  NMVOC_e      mean=    0.0000 std=    0.0000
  NMVOC_finn   mean=    0.0000 std=    0.0000
  bio          mean=    0.0000 std=    0.0000


Saved episode mask for APRIL_16 -> /kaggle/working/experiments/phase2_competent/episode_masks/APRIL_16_episode_mask.npy | ratio=0.02078


Saved episode mask for JULY_16 -> /kaggle/working/experiments/phase2_competent/episode_masks/JULY_16_episode_mask.npy | ratio=0.01973


Saved episode mask for OCT_16 -> /kaggle/working/experiments/phase2_competent/episode_masks/OCT_16_episode_mask.npy | ratio=0.01704


Saved episode mask for DEC_16 -> /kaggle/working/experiments/phase2_competent/episode_masks/DEC_16_episode_mask.npy | ratio=0.02173
Timeline tensor shape: (2932, 16, 140, 124) | RAM ~ 3.26 GB
Episode mask shape   : (2932, 140, 124) | ratio=0.01981
Train samples        : 2446
Val samples          : 436
Train batches        : 1223
Val batches          : 436
hist batch   : (2, 10, 16, 140, 124) (B, T_in, V, H, W)
target batch : (2, 16, 140, 124) (B, T_out, H, W)
mask batch   : (2, 16, 140, 124) (B, T_out, H, W)


In [4]:
# -----------------------------
# Model + loss + metric helpers
# -----------------------------
class ConvLSTMCell(nn.Module):
    def __init__(self, in_ch, hidden_ch, ks=3):
        super().__init__()
        self.hidden_ch = hidden_ch
        self.conv = nn.Conv2d(in_ch + hidden_ch, 4 * hidden_ch, ks, padding=ks // 2, bias=True)

    def forward(self, x, h, c):
        gates = self.conv(torch.cat([x, h], dim=1))
        i, f, o, g = torch.chunk(gates, 4, dim=1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        o = torch.sigmoid(o)
        g = torch.tanh(g)
        c = f * c + i * g
        h = o * torch.tanh(c)
        return h, c

    def init_hidden(self, b, h, w, dev):
        z = torch.zeros(b, self.hidden_ch, h, w, device=dev)
        return z, z.clone()


class ConvLSTM(nn.Module):
    def __init__(self, in_ch, hidden_ch, n_layers=2):
        super().__init__()
        self.cells = nn.ModuleList([
            ConvLSTMCell(in_ch if i == 0 else hidden_ch, hidden_ch)
            for i in range(n_layers)
        ])

    def forward(self, seq):
        b, t, c, h, w = seq.shape
        states = [cell.init_hidden(b, h, w, seq.device) for cell in self.cells]

        for ti in range(t):
            x = seq[:, ti]
            for li, cell in enumerate(self.cells):
                h_i, c_i = states[li]
                h_i, c_i = cell(x, h_i, c_i)
                states[li] = (h_i, c_i)
                x = h_i

        return states[-1][0]


class TemporalAttention(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_ch, max(4, in_ch // 4)),
            nn.GELU(),
            nn.Linear(max(4, in_ch // 4), 1),
        )

    def forward(self, x):
        # x: (B, T, C, H, W)
        scores = self.mlp(x.mean(dim=(-1, -2)))
        w = torch.softmax(scores, dim=1)
        return (x * w.unsqueeze(-1).unsqueeze(-1)).sum(dim=1)


class ResConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout_p=0.0):
        super().__init__()
        drop = nn.Dropout2d(dropout_p) if dropout_p > 0 else nn.Identity()
        self.body = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
            drop,
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
        )
        self.short = nn.Identity() if in_ch == out_ch else nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
        )
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(self.body(x) + self.short(x))


class EpisodeAwareUNet(nn.Module):
    def __init__(
        self,
        v=V,
        time_in=TIME_IN,
        time_out=TIME_OUT,
        base_ch=UNET_BASE_CH,
        lstm_hidden=LSTM_HIDDEN,
        n_levels=N_LEVELS,
        dropout_p=DROPOUT,
    ):
        super().__init__()
        self.time_out = time_out

        self.feat_proj = nn.Sequential(
            nn.Conv2d(v, base_ch, 1, bias=False),
            nn.BatchNorm2d(base_ch),
            nn.GELU(),
        )
        self.conv_lstm = ConvLSTM(base_ch, lstm_hidden, n_layers=2)
        self.temp_attn = TemporalAttention(base_ch)
        self.hist_fuse = nn.Sequential(
            nn.Conv2d(lstm_hidden + base_ch, base_ch, 1, bias=False),
            nn.BatchNorm2d(base_ch),
            nn.GELU(),
        )

        self.enc_blocks = nn.ModuleList()
        self.pools = nn.ModuleList()
        self.skip_ch = []

        ch = base_ch
        for _ in range(n_levels):
            out_ch = ch * 2
            self.enc_blocks.append(ResConvBlock(ch, out_ch, dropout_p))
            self.pools.append(nn.MaxPool2d(2))
            self.skip_ch.append(out_ch)
            ch = out_ch

        bot_ch = ch * 2
        self.bottleneck = nn.Sequential(
            ResConvBlock(ch, bot_ch, dropout_p),
            ResConvBlock(bot_ch, bot_ch, dropout_p),
        )
        ch = bot_ch

        self.up = nn.ModuleList()
        self.dec = nn.ModuleList()
        for s_ch in reversed(self.skip_ch):
            up_out = ch // 2
            self.up.append(nn.ConvTranspose2d(ch, up_out, 2, stride=2))
            self.dec.append(ResConvBlock(up_out + s_ch, up_out, dropout_p))
            ch = up_out

        self.head = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1, bias=False),
            nn.GELU(),
            nn.Conv2d(ch, time_out, 1),
        )

    def forward(self, hist):
        # hist: (B, T_in, V, H, W)
        b, t, v, h, w = hist.shape
        x = self.feat_proj(hist.reshape(b * t, v, h, w)).reshape(b, t, -1, h, w)

        h_lstm = self.conv_lstm(x)
        h_attn = self.temp_attn(x)
        feat = self.hist_fuse(torch.cat([h_lstm, h_attn], dim=1))

        skips = []
        for enc, pool in zip(self.enc_blocks, self.pools):
            feat = enc(feat)
            skips.append(feat)
            feat = pool(feat)

        feat = self.bottleneck(feat)

        for up, dec, sk in zip(self.up, self.dec, reversed(skips)):
            feat = up(feat)
            if feat.shape[-2:] != sk.shape[-2:]:
                feat = F.interpolate(feat, size=sk.shape[-2:], mode="bilinear", align_corners=False)
            feat = dec(torch.cat([feat, sk], dim=1))

        last_pm25 = hist[:, -1, 0:1]
        out = self.head(feat) + last_pm25
        return out


def denorm_pm25_torch(x):
    return x * PM_STD + PM_MEAN


def smape_torch(pred, tgt, eps=1e-6):
    den = 0.5 * (pred.abs() + tgt.abs()) + eps
    return (pred - tgt).abs() / den


class EpisodeAwareLoss(nn.Module):
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps

    def forward(self, pred_z, tgt_z, ep_mask):
        pred = F.relu(denorm_pm25_torch(pred_z))
        tgt = denorm_pm25_torch(tgt_z)

        global_rmse = torch.sqrt(torch.mean((pred - tgt) ** 2) + self.eps)

        smape_map = smape_torch(pred, tgt, eps=self.eps)
        global_smape = smape_map.mean()

        m = (ep_mask > 0.5).float()
        m_sum = m.sum()
        if float(m_sum.detach().cpu()) > 0:
            episode_smape = (smape_map * m).sum() / (m_sum + self.eps)
        else:
            episode_smape = global_smape

        corr_vals = []
        bsz, tout = pred.shape[0], pred.shape[1]
        for bi in range(bsz):
            for ti in range(tout):
                mt = m[bi, ti] > 0.5
                if int(mt.sum().item()) < 4:
                    continue
                p = pred[bi, ti][mt]
                y = tgt[bi, ti][mt]
                p = p - p.mean()
                y = y - y.mean()
                den = torch.sqrt((p.pow(2).sum() + self.eps) * (y.pow(2).sum() + self.eps))
                corr_vals.append((p * y).sum() / den)

        episode_corr = torch.stack(corr_vals).mean() if len(corr_vals) > 0 else pred.new_tensor(0.0)

        # Penalize positive bias where mask says "non-episode".
        non_ep = 1.0 - m
        non_ep_sum = non_ep.sum()
        over = torch.relu(pred - tgt)
        if float(non_ep_sum.detach().cpu()) > 0:
            non_ep_overshoot = (over * non_ep).sum() / (non_ep_sum + self.eps)
        else:
            non_ep_overshoot = pred.new_tensor(0.0)

        loss = (
            W_RMSE * global_rmse
            + W_GLOBAL_SMAPE * global_smape
            + W_EPISODE_SMAPE * episode_smape
            + W_EPISODE_CORR * (1.0 - episode_corr)
            + W_NON_EP_OVERSHOOT * non_ep_overshoot
        )

        metrics = {
            "global_rmse": float(global_rmse.detach().cpu()),
            "global_smape": float(global_smape.detach().cpu()),
            "episode_smape": float(episode_smape.detach().cpu()),
            "episode_corr": float(episode_corr.detach().cpu()),
            "non_ep_overshoot": float(non_ep_overshoot.detach().cpu()),
        }
        return loss, metrics


def _safe_corr(a, b, eps=1e-8):
    if a.size < 4:
        return 0.0
    a = a.astype(np.float64)
    b = b.astype(np.float64)
    a = a - a.mean()
    b = b - b.mean()
    den = np.sqrt((a * a).sum() * (b * b).sum()) + eps
    return float((a * b).sum() / den)


def compute_phase2_metrics_np(y_true, y_pred, episode_mask, eps=1e-6):
    # all inputs: (N, H, W, T)
    t_out = y_true.shape[-1]

    global_smape_ts = []
    episode_smape_ts = []
    episode_corr_ts = []

    for t in range(t_out):
        yt = y_true[..., t]
        yp = y_pred[..., t]
        mt = episode_mask[..., t] > 0.5

        g_smape_t = np.mean(np.abs(yp - yt) / (0.5 * (np.abs(yt) + np.abs(yp)) + eps))
        global_smape_ts.append(float(g_smape_t))

        if mt.any():
            ya = yt[mt]
            pa = yp[mt]
            e_smape_t = np.mean(np.abs(pa - ya) / (0.5 * (np.abs(ya) + np.abs(pa)) + eps))
            e_corr_t = _safe_corr(ya, pa)
        else:
            e_smape_t = float(g_smape_t)
            e_corr_t = 0.0

        episode_smape_ts.append(float(e_smape_t))
        episode_corr_ts.append(float(e_corr_t))

    global_smape = float(np.mean(global_smape_ts))
    episode_smape = float(np.mean(episode_smape_ts))
    episode_corr = float(np.mean(episode_corr_ts))

    norm_episode_corr = (episode_corr + 1.0) / 2.0
    norm_global_smape = 1.0 - (global_smape / 2.0)
    norm_episode_smape = 1.0 - (episode_smape / 2.0)

    equal_weight_score = float((norm_episode_corr + norm_global_smape + norm_episode_smape) / 3.0)

    return {
        "global_smape": global_smape,
        "episode_smape": episode_smape,
        "episode_corr": episode_corr,
        "norm_episode_corr": norm_episode_corr,
        "norm_global_smape": norm_global_smape,
        "norm_episode_smape": norm_episode_smape,
        "equal_weight_score": equal_weight_score,
    }


model = EpisodeAwareUNet().to(device)
loss_fn = EpisodeAwareLoss().to(device)

print(f"Model parameters: {count_params(model):,}")
print("Model + loss ready")

Model parameters: 5,209,157
Model + loss ready


In [5]:
# -----------------------------
# Training loop
# -----------------------------
def _build_optimizer_scheduler(max_lr, n_epochs):
    opt = torch.optim.AdamW(model.parameters(), lr=max_lr, weight_decay=WEIGHT_DECAY)
    steps = max(1, math.ceil(len(train_loader) / max(1, GRAD_ACCUM_STEPS)))
    sch = torch.optim.lr_scheduler.OneCycleLR(
        opt,
        max_lr=max_lr,
        epochs=n_epochs,
        steps_per_epoch=steps,
        pct_start=0.30,
        anneal_strategy="cos",
        div_factor=10.0,
        final_div_factor=1e2,
    )
    return opt, sch, steps


optimizer, scheduler, optim_steps_per_epoch = _build_optimizer_scheduler(LR, EPOCHS)
scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)

LATEST_CKPT = os.path.join(CKPT_DIR, "phase2_latest.pt")
BEST_CKPT = os.path.join(CKPT_DIR, "phase2_best.pt")


def _save_state(path, epoch, best_val_obj, val_obj, val_metrics, patience_counter, topk_records, log_history):
    save_checkpoint(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict() if AMP_ENABLED else None,
            "best_val_obj": float(best_val_obj),
            "val_obj": float(val_obj),
            "val_metrics": val_metrics,
            "patience_counter": int(patience_counter),
            "topk_records": topk_records,
            "log_history": log_history,
            "data_stats": DATA_STATS,
            "config": {
                "TIME_IN": TIME_IN,
                "TIME_OUT": TIME_OUT,
                "UNET_BASE_CH": UNET_BASE_CH,
                "LSTM_HIDDEN": LSTM_HIDDEN,
                "N_LEVELS": N_LEVELS,
                "W_RMSE": W_RMSE,
                "W_GLOBAL_SMAPE": W_GLOBAL_SMAPE,
                "W_EPISODE_SMAPE": W_EPISODE_SMAPE,
                "W_EPISODE_CORR": W_EPISODE_CORR,
                "W_NON_EP_OVERSHOOT": W_NON_EP_OVERSHOOT,
            },
        },
        path,
    )


def run_train_epoch():
    model.train()
    optimizer.zero_grad(set_to_none=True)

    loss_sum = 0.0

    pbar = tqdm(train_loader, desc="Train", leave=False, dynamic_ncols=True)
    for step_idx, (hist_b, targ_b, msk_b) in enumerate(pbar):
        hist_b = move_to_device(hist_b)
        targ_b = move_to_device(targ_b)
        msk_b = move_to_device(msk_b)

        amp_ctx = torch.cuda.amp.autocast(enabled=AMP_ENABLED) if AMP_ENABLED else nullcontext()
        with amp_ctx:
            pred_b = model(hist_b)
            loss_b, metrics_b = loss_fn(pred_b, targ_b, msk_b)

        loss_step = loss_b / max(1, GRAD_ACCUM_STEPS)

        if AMP_ENABLED:
            scaler.scale(loss_step).backward()
        else:
            loss_step.backward()

        do_step = ((step_idx + 1) % max(1, GRAD_ACCUM_STEPS) == 0) or ((step_idx + 1) == len(train_loader))
        if do_step:
            if AMP_ENABLED:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                if IS_TPU:
                    xm.optimizer_step(optimizer, barrier=False)
                else:
                    optimizer.step()

            if scheduler.last_epoch + 1 < scheduler.total_steps:
                scheduler.step()

            optimizer.zero_grad(set_to_none=True)
            maybe_mark_step()

        loss_sum += float(loss_b.detach().cpu())
        pbar.set_postfix(
            loss=f"{float(loss_b.detach().cpu()):.4f}",
            rmse=f"{metrics_b['global_rmse']:.3f}",
            nep=f"{metrics_b['non_ep_overshoot']:.3f}",
        )

    maybe_mark_step()
    return loss_sum / max(1, len(train_loader))


def run_val_epoch():
    model.eval()
    loss_sum = 0.0

    pred_all = []
    true_all = []
    mask_all = []

    with torch.no_grad():
        pbar = tqdm(val_loader, desc="Val", leave=False, dynamic_ncols=True)
        for step_idx, (hist_b, targ_b, msk_b) in enumerate(pbar):
            hist_b = move_to_device(hist_b)
            targ_b = move_to_device(targ_b)
            msk_b = move_to_device(msk_b)

            pred_b = model(hist_b)
            loss_b, _ = loss_fn(pred_b, targ_b, msk_b)
            loss_sum += float(loss_b.detach().cpu())

            pred_phys = F.relu(denorm_pm25_torch(pred_b)).detach().cpu().numpy().transpose(0, 2, 3, 1)
            true_phys = denorm_pm25_torch(targ_b).detach().cpu().numpy().transpose(0, 2, 3, 1)
            mask_np = msk_b.detach().cpu().numpy().transpose(0, 2, 3, 1)

            pred_all.append(pred_phys)
            true_all.append(true_phys)
            mask_all.append(mask_np)

            if (step_idx + 1) % 20 == 0:
                maybe_mark_step()

    maybe_mark_step()

    pred_np = np.concatenate(pred_all, axis=0)
    true_np = np.concatenate(true_all, axis=0)
    mask_np = np.concatenate(mask_all, axis=0)

    phase2_metrics = compute_phase2_metrics_np(true_np, pred_np, mask_np)

    # Lower is better; proxy aligns with hidden metric components.
    val_obj = (
        0.55 * phase2_metrics["global_smape"]
        + 0.30 * phase2_metrics["episode_smape"]
        + 0.15 * (1.0 - phase2_metrics["episode_corr"])
    )

    return loss_sum / max(1, len(val_loader)), phase2_metrics, float(val_obj)


def _persist_logs(topk_records, log_history):
    with open(TRAIN_LOG_JSON, "w") as f:
        json.dump(log_history, f, indent=2)
    with open(TOPK_JSON, "w") as f:
        json.dump(topk_records, f, indent=2)


def _update_topk(epoch, val_obj, val_metrics, best_val_obj, patience_counter, topk_records, log_history):
    candidate = {
        "epoch": int(epoch),
        "val_obj": float(val_obj),
        "global_smape": float(val_metrics["global_smape"]),
        "episode_smape": float(val_metrics["episode_smape"]),
        "episode_corr": float(val_metrics["episode_corr"]),
        "equal_weight_score": float(val_metrics["equal_weight_score"]),
        "path": os.path.join(CKPT_DIR, f"phase2_topk_e{epoch:03d}.pt"),
    }

    if (len(topk_records) < TOPK_BEST) or (val_obj < max(x["val_obj"] for x in topk_records)):
        _save_state(
            candidate["path"],
            epoch,
            best_val_obj,
            val_obj,
            val_metrics,
            patience_counter,
            topk_records,
            log_history,
        )

        topk_records.append(candidate)
        topk_records = sorted(topk_records, key=lambda x: x["val_obj"])[:TOPK_BEST]

        kept = set(x["path"] for x in topk_records)
        for fn in os.listdir(CKPT_DIR):
            if fn.startswith("phase2_topk_e"):
                fp = os.path.join(CKPT_DIR, fn)
                if fp not in kept:
                    try:
                        os.remove(fp)
                    except Exception:
                        pass

    return topk_records


start_epoch = 1
best_val_obj = float("inf")
patience_counter = 0
topk_records = []
log_history = []

if RESUME_IF_AVAILABLE and os.path.exists(LATEST_CKPT):
    ckpt = load_checkpoint(LATEST_CKPT)
    try:
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        if AMP_ENABLED and ckpt.get("scaler_state_dict") is not None:
            scaler.load_state_dict(ckpt["scaler_state_dict"])

        start_epoch = int(ckpt.get("epoch", 0)) + 1
        best_val_obj = float(ckpt.get("best_val_obj", float("inf")))
        patience_counter = int(ckpt.get("patience_counter", 0))
        topk_records = ckpt.get("topk_records", [])
        log_history = ckpt.get("log_history", [])

        print(f"Resumed from {LATEST_CKPT} at epoch {start_epoch}")
    except Exception as e:
        print(f"Resume skipped due to load mismatch: {e}")

print(f"\nStage-1 training starts | epochs={EPOCHS} | start_epoch={start_epoch}")
print(f"Columns: epoch | train_loss | val_loss | val_obj | gSMAPE | eSMAPE | eCorr | eqScore | lr | sec")

train_t0 = time.time()
last_stage1_epoch = start_epoch - 1

for epoch in range(start_epoch, EPOCHS + 1):
    ep_t0 = time.time()

    train_loss = run_train_epoch()
    val_loss, val_metrics, val_obj = run_val_epoch()

    lr_now = float(optimizer.param_groups[0]["lr"])
    ep_sec = time.time() - ep_t0

    log_entry = {
        "phase": "stage1",
        "epoch": int(epoch),
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "val_obj": float(val_obj),
        "val_global_smape": float(val_metrics["global_smape"]),
        "val_episode_smape": float(val_metrics["episode_smape"]),
        "val_episode_corr": float(val_metrics["episode_corr"]),
        "val_equal_weight_score": float(val_metrics["equal_weight_score"]),
        "lr": lr_now,
        "sec": float(ep_sec),
    }
    log_history.append(log_entry)

    is_best = val_obj < best_val_obj
    if is_best:
        best_val_obj = val_obj
        patience_counter = 0
        _save_state(
            BEST_CKPT,
            epoch,
            best_val_obj,
            val_obj,
            val_metrics,
            patience_counter,
            topk_records,
            log_history,
        )
    else:
        patience_counter += 1

    _save_state(
        LATEST_CKPT,
        epoch,
        best_val_obj,
        val_obj,
        val_metrics,
        patience_counter,
        topk_records,
        log_history,
    )

    topk_records = _update_topk(epoch, val_obj, val_metrics, best_val_obj, patience_counter, topk_records, log_history)
    _persist_logs(topk_records, log_history)

    print(
        f"[{epoch:03d}/{EPOCHS}] "
        f"{train_loss:8.4f} | {val_loss:8.4f} | {val_obj:8.4f} | "
        f"{val_metrics['global_smape']:.4f} | {val_metrics['episode_smape']:.4f} | "
        f"{val_metrics['episode_corr']:.4f} | {val_metrics['equal_weight_score']:.4f} | "
        f"{lr_now:.2e} | {ep_sec:.0f}s"
    )

    last_stage1_epoch = epoch
    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f"Early stopping triggered in stage-1: no val_obj improvement for {EARLY_STOP_PATIENCE} epochs")
        break


# -----------------------------
# Stage-2 fine-tune from best checkpoint
# -----------------------------
if ENABLE_STAGE2_FINETUNE and STAGE2_EPOCHS > 0 and os.path.exists(BEST_CKPT):
    print("\nStarting stage-2 fine-tune from best checkpoint...")

    best_ck = load_checkpoint(BEST_CKPT)
    model.load_state_dict(best_ck["model_state_dict"])

    optimizer, scheduler, _ = _build_optimizer_scheduler(STAGE2_LR, STAGE2_EPOCHS)
    if AMP_ENABLED:
        scaler = torch.cuda.amp.GradScaler(enabled=True)

    ft_no_improve = 0
    stage2_start_epoch = max(last_stage1_epoch + 1, EPOCHS + 1)

    for ft_i in range(1, STAGE2_EPOCHS + 1):
        epoch = stage2_start_epoch + ft_i - 1
        ep_t0 = time.time()

        train_loss = run_train_epoch()
        val_loss, val_metrics, val_obj = run_val_epoch()

        lr_now = float(optimizer.param_groups[0]["lr"])
        ep_sec = time.time() - ep_t0

        log_entry = {
            "phase": "stage2",
            "epoch": int(epoch),
            "train_loss": float(train_loss),
            "val_loss": float(val_loss),
            "val_obj": float(val_obj),
            "val_global_smape": float(val_metrics["global_smape"]),
            "val_episode_smape": float(val_metrics["episode_smape"]),
            "val_episode_corr": float(val_metrics["episode_corr"]),
            "val_equal_weight_score": float(val_metrics["equal_weight_score"]),
            "lr": lr_now,
            "sec": float(ep_sec),
        }
        log_history.append(log_entry)

        is_best = val_obj < best_val_obj
        if is_best:
            best_val_obj = val_obj
            ft_no_improve = 0
            patience_counter = 0
            _save_state(
                BEST_CKPT,
                epoch,
                best_val_obj,
                val_obj,
                val_metrics,
                patience_counter,
                topk_records,
                log_history,
            )
        else:
            ft_no_improve += 1
            patience_counter = ft_no_improve

        _save_state(
            LATEST_CKPT,
            epoch,
            best_val_obj,
            val_obj,
            val_metrics,
            patience_counter,
            topk_records,
            log_history,
        )

        topk_records = _update_topk(epoch, val_obj, val_metrics, best_val_obj, patience_counter, topk_records, log_history)
        _persist_logs(topk_records, log_history)

        print(
            f"[FT{ft_i:02d}/{STAGE2_EPOCHS}] "
            f"{train_loss:8.4f} | {val_loss:8.4f} | {val_obj:8.4f} | "
            f"{val_metrics['global_smape']:.4f} | {val_metrics['episode_smape']:.4f} | "
            f"{val_metrics['episode_corr']:.4f} | {val_metrics['equal_weight_score']:.4f} | "
            f"{lr_now:.2e} | {ep_sec:.0f}s"
        )

        if ft_no_improve >= STAGE2_PATIENCE:
            print(f"Stage-2 early stopping: no val_obj improvement for {STAGE2_PATIENCE} epochs")
            break


train_min = (time.time() - train_t0) / 60.0
print("-" * 100)
print(f"Training complete in {train_min:.1f} min")
print(f"Best val_obj          : {best_val_obj:.6f}")
print(f"Best checkpoint       : {BEST_CKPT}")
print(f"Latest checkpoint     : {LATEST_CKPT}")
print(f"Top-k metadata saved  : {TOPK_JSON}")


Stage-1 training starts | epochs=40 | start_epoch=1
Columns: epoch | train_loss | val_loss | val_obj | gSMAPE | eSMAPE | eCorr | eqScore | lr | sec


[001/40]   6.5490 |   9.9189 |   0.3650 | 0.3888 | 0.4658 | 0.9238 | 0.8449 | 8.07e-05 | 217s


[002/40]   5.7784 |  10.4898 |   0.3642 | 0.3794 | 0.4798 | 0.9224 | 0.8439 | 1.12e-04 | 203s


[003/40]   5.4591 |   9.3314 |   0.3372 | 0.3446 | 0.4567 | 0.9286 | 0.8545 | 1.62e-04 | 206s


[004/40]   5.2446 |   9.2511 |   0.3063 | 0.3235 | 0.3960 | 0.9364 | 0.8695 | 2.28e-04 | 204s


...

[FT03/8]   3.0049 |   7.8744 |   0.2421 | 0.2403 | 0.3363 | 0.9393 | 0.8938 | 1.94e-04 | 205s


[FT04/8]   2.9577 |   7.7654 |   0.2436 | 0.2410 | 0.3405 | 0.9404 | 0.8932 | 1.62e-04 | 205s


[FT05/8]   2.9096 |   7.7189 |   0.2436 | 0.2386 | 0.3453 | 0.9416 | 0.8930 | 1.11e-04 | 204s


[FT06/8]   2.8551 |   7.8521 |   0.2788 | 0.2763 | 0.3920 | 0.9383 | 0.8783 | 5.66e-05 | 205s


[FT07/8]   2.8219 |   7.7400 |   0.2449 | 0.2395 | 0.3469 | 0.9395 | 0.8922 | 1.54e-05 | 205s
Stage-2 early stopping: no val_obj improvement for 4 epochs
----------------------------------------------------------------------------------------------------
Training complete in 116.3 min
Best val_obj          : 0.242137
Best checkpoint       : /kaggle/working/experiments/phase2_competent/checkpoints/phase2_best.pt
Latest checkpoint     : /kaggle/working/experiments/phase2_competent/checkpoints/phase2_latest.pt
Top-k metadata saved  : /kaggle/working/experiments/phase2_competent/logs/topk_checkpoints.json


In [6]:
# -----------------------------
# Test inference + weighted top-k ensemble
# -----------------------------
try:
    del train_ds, val_ds, train_loader, val_loader
except Exception:
    pass
gc.collect()
if DEVICE_KIND == "cuda":
    torch.cuda.empty_cache()


def _canon_4d(arr, feat_name):
    arr = np.asarray(arr)
    if arr.ndim != 4:
        raise ValueError(f"[{feat_name}] expected 4D, got {arr.shape}")

    # Preferred: (N, T, H, W)
    if arr.shape[2] == S1 and arr.shape[3] == S2:
        return arr.astype(np.float32, copy=False)

    # Alternate: (N, H, W, T)
    if arr.shape[1] == S1 and arr.shape[2] == S2:
        return np.transpose(arr, (0, 3, 1, 2)).astype(np.float32, copy=False)

    raise ValueError(f"[{feat_name}] unknown layout {arr.shape}")


class TestDataset(Dataset):
    def __init__(self):
        feat_data = {}
        for feat in ALL_FEATS:
            fp = os.path.join(TEST_IN_PATH, f"{feat}.npy")
            feat_data[feat] = _canon_4d(np.load(fp), feat)

        n_min = min(x.shape[0] for x in feat_data.values())
        h_min = min(x.shape[2] for x in feat_data.values())
        w_min = min(x.shape[3] for x in feat_data.values())

        if h_min < S1 or w_min < S2:
            raise ValueError(f"Test spatial shape too small. min=({h_min},{w_min}) expected=({S1},{S2})")

        for feat, arr in feat_data.items():
            if arr.shape[1] < TIME_IN:
                raise ValueError(f"{feat} needs >= TIME_IN={TIME_IN}, got {arr.shape[1]}")

        hist_feats = []
        for feat in ALL_FEATS:
            arr = feat_data[feat][:n_min, :TIME_IN, :S1, :S2].astype(np.float32)
            arr = (arr - float(DATA_STATS[feat]["mean"])) / float(DATA_STATS[feat]["std"])
            hist_feats.append(arr)

        hist_np = np.stack(hist_feats, axis=2).astype(np.float32)  # (N, T_in, V, H, W)
        self.hist = torch.from_numpy(hist_np)
        self.N = n_min

        print(f"Test samples: {self.N} | hist shape: {tuple(self.hist.shape)}")

    def __len__(self):
        return self.N

    def __getitem__(self, idx):
        return self.hist[idx].contiguous()


def _predict_with_checkpoint(ckpt_path, loader, use_tta=True):
    ckpt = load_checkpoint(ckpt_path)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    all_preds = []
    with torch.no_grad():
        for step_idx, hist_b in enumerate(tqdm(loader, desc=f"Infer {os.path.basename(ckpt_path)}", leave=False)):
            hist_b = move_to_device(hist_b)

            pred0 = model(hist_b)
            if use_tta:
                hist_flip = hist_b.flip(-1).clone()
                # Horizontal mirror changes zonal wind direction; fix u10 sign.
                hist_flip[:, :, U10_IDX] = -hist_flip[:, :, U10_IDX]
                pred1 = model(hist_flip).flip(-1)
                pred = 0.5 * (pred0 + pred1)
            else:
                pred = pred0

            pred = F.relu(denorm_pm25_torch(pred)).detach().cpu().numpy()  # (B, T_out, H, W)
            all_preds.append(pred.transpose(0, 2, 3, 1))

            if (step_idx + 1) % 20 == 0:
                maybe_mark_step()

    maybe_mark_step()
    out = np.concatenate(all_preds, axis=0)
    val_obj = float(ckpt.get("val_obj", np.nan))
    epoch = int(ckpt.get("epoch", -1))
    return out, val_obj, epoch


test_ds = TestDataset()
test_loader = DataLoader(
    test_ds,
    batch_size=TEST_BS,
    shuffle=False,
    num_workers=0 if IS_TPU else 2,
    pin_memory=PIN_MEMORY,
    persistent_workers=(not IS_TPU) and (2 > 0),
)

ckpt_entries = []
if os.path.exists(TOPK_JSON):
    with open(TOPK_JSON, "r") as f:
        loaded = json.load(f)
    for x in loaded:
        if os.path.exists(x["path"]):
            ckpt_entries.append(x)

if len(ckpt_entries) == 0:
    fallback = BEST_CKPT if os.path.exists(BEST_CKPT) else LATEST_CKPT
    if not os.path.exists(fallback):
        raise FileNotFoundError("No checkpoint available for inference. Train first or provide checkpoints.")
    ckpt_entries = [{"path": fallback, "val_obj": 1e9, "epoch": -1}]
    print("Top-k metadata missing. Falling back to best/latest checkpoint.")

# Lower val_obj is better -> weight by inverse val_obj.
val_objs = np.array([max(1e-6, float(x.get("val_obj", 1e9))) for x in ckpt_entries], dtype=np.float64)
weights = 1.0 / val_objs
weights = weights / weights.sum()

print("Ensemble checkpoints:")
for x, w in zip(ckpt_entries, weights):
    print(f"  epoch={int(x.get('epoch', -1)):>3} val_obj={float(x.get('val_obj', np.nan)):.6f} weight={w:.4f} file={os.path.basename(x['path'])}")

submission = None
for i, x in enumerate(ckpt_entries):
    pred_i, val_obj_i, epoch_i = _predict_with_checkpoint(x["path"], test_loader, use_tta=USE_FLIP_TTA)
    w = float(weights[i])
    submission = pred_i * w if submission is None else submission + pred_i * w

submission = np.clip(submission, 0.0, None).astype(np.float32)
np.save(PRED_SAVE_PATH, submission)

print(f"Saved: {PRED_SAVE_PATH}")
print(f"Prediction shape: {submission.shape}")
print(f"Value range: [{submission.min():.3f}, {submission.max():.3f}] ug/m3")

expected_shape = (218, S1, S2, TIME_OUT)
if submission.shape != expected_shape:
    print(f"[WARN] Expected shape {expected_shape}, got {submission.shape}")
else:
    print("Submission shape is valid for Phase 2")

Test samples: 218 | hist shape: (218, 10, 16, 140, 124)
Ensemble checkpoints:
  epoch= 43 val_obj=0.242137 weight=0.3347 file=phase2_topk_e043.pt
  epoch= 45 val_obj=0.243558 weight=0.3327 file=phase2_topk_e045.pt
  epoch= 44 val_obj=0.243650 weight=0.3326 file=phase2_topk_e044.pt


Saved: /kaggle/working/preds.npy
Prediction shape: (218, 140, 124, 16)
Value range: [0.000, 1416.588] ug/m3
Submission shape is valid for Phase 2
